# Saved (named) LLM configs

A **saved LLM config** is a team-scoped, named record wrapping either a
single provider config or an LLM group. Instead of inlining provider
settings (and API keys) into every node, you register the config once
and reference it from a node via `named_llm_config_id` /
`named_llm_config_name`.

Everything lives under `client.llm_configs`: `schema`, `create`,
`list`, `get_default`, `get`, `update`, `delete`, `test`, `test_inline`.

See the companion guide: [`../docs/guides/llm_configs.md`](../docs/guides/llm_configs.md).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient
from interactly.configs import OpenAILLMConfig, OPENAIModel

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

In [ ]:
# Idempotency guard — remove anything left over from a previous run so that
# re-running this notebook always starts from a clean slate (config names must be unique).
_demo_names = {"Primary GPT config", "Primary GPT config (v2)"}
_existing = await (await client.llm_configs.list(size=200)).list_all()
for _c in _existing:
    if _c.name in _demo_names:
        await client.llm_configs.delete(_c.id)

# Also remove the demo workflow this notebook builds at the end (see "Referencing a
# saved config from a node"), in case a prior run was interrupted before cleanup.
_demo_wf_name = "Saved-config demo (10_llm_configs)"
for _w in await (await client.workflows.list(size=200)).list_all():
    if _w.name == _demo_wf_name:
        await client.workflows.delete(_w.id)

print("Cleared any pre-existing demo assets.")

## Discover the schema

`schema()` returns the JSON Schema for a saved config's `config` field
(an `LLMOrGroupConfig`).

In [ ]:
from typing import Dict, Any

schema: Dict[str, Any] = await client.llm_configs.schema()
print(sorted(schema.keys()))

## Create a saved config

`create()` takes a keyword-only `name` and `config`. The `config` can be
a typed `OpenAILLMConfig` (or any `LLMOrGroupConfig`) or a compatible
dict. Set `is_default=True` to mark it as the team default; pair with
`override_default=True` to replace an existing default.

In [ ]:
cfg = OpenAILLMConfig(
    model=OPENAIModel.GPT_5_4,
    max_tokens=300,
    temperature=0.2,
)

saved = await client.llm_configs.create(
    name="Primary GPT config",
    config=cfg,
    description="Default OpenAI settings for assistant nodes",
    is_default=True,
    override_default=True,
)
print(saved.id, saved.name, saved.is_default)

## List, get, and fetch the default

In [ ]:
# Paginated list with optional fuzzy `search`.
page = await client.llm_configs.list(size=10, search="GPT")
for item in page.items:
    print(item.id, item.name, item.is_default)

# Fetch a specific config by id.
fetched = await client.llm_configs.get(saved.id)

# Fetch whichever config the team has marked as default.
default_cfg = await client.llm_configs.get_default()
print(default_cfg.id, default_cfg.name)

## Update

`update()` sends only the fields you pass. Note the provider `api_key`
is redacted by the server on read, so re-sending a fetched config reuses
the stored secret.

In [ ]:
updated = await client.llm_configs.update(
    saved.id,
    name="Primary GPT config (v2)",
    description="Bumped temperature",
    config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4, max_tokens=300, temperature=0.4),
)
print(updated.name)

## Test a config

Two flavours:

- `test(id, ...)` exercises a **saved** config against a `system_prompt`
  (+ optional `messages`); a redacted inline `config` override reuses the
  stored secret.
- `test_inline(...)` exercises an **unsaved** config before persisting —
  the `config` must carry its own `api_key` (or omit it to fall back to
  the team's stored vendor credentials).

Both return an `LLMConfigTestResult` (`success`, `response`, token counts,
`latency_ms`, plus group extras like `winning_member`).

In [ ]:
# Test the saved config.
result = await client.llm_configs.test(
    saved.id,
    system_prompt="You are a terse assistant. Answer in one short sentence.",
    messages=[{"role": "human", "content": "What is an insurance deductible?"}],
)
print(result.success, result.response)
print(result.provider, result.model, result.total_tokens, result.latency_ms)

# Test an inline (unsaved) config before you decide to persist it.
inline_result = await client.llm_configs.test_inline(
    system_prompt="You are a terse assistant.",
    config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4_NANO, max_tokens=30),
    messages=[{"role": "human", "content": "Say hello."}],
)
print(inline_result.success, inline_result.response)

## Referencing a saved config from a node

Once a config is saved, point a node's `llms_config` at it by id instead
of inlining provider settings. Every `BaseLLMConfig` carries
`named_llm_config_id` (and `named_llm_config_name`) for exactly this:

```python
from interactly.configs import SayLLMNodeConfig, PromptConfig, OpenAILLMConfig

node = SayLLMNodeConfig(
    name="Assistant",
    wait_for_user_message=True,
    main_response_config=PromptConfig(prompt="You are a helpful assistant."),
    # Resolve provider settings from the saved config instead of inlining them:
    llms_config=OpenAILLMConfig(named_llm_config_id=saved.id),
)
```

The server resolves the reference at run time and fills in the stored
provider/model/secret.

In [ ]:
# Build a single-LLM-node workflow whose node resolves its provider settings from the
# SAVED config (by id) — then run it to prove the reference works end to end.
from langchain_core.messages import HumanMessage

from interactly.configs import (
    SayLLMNodeConfig,
    PromptConfig,
    OpenAILLMConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
    LLMNodeRunInput,
    NodesRunInputs,
    WorkflowCommand,
    WorkflowRunInput,
)
from interactly.types.workflows.workflow import Workflow
from interactly.runtime.events import AssistantResponseEvent

# The node inlines NO provider settings — only a pointer to the saved config.
assistant_node = SayLLMNodeConfig(
    name="Assistant",
    is_start=True,
    wait_for_user_message=True,
    main_response_config=PromptConfig(
        prompt="You are a helpful health-insurance assistant. Answer the user's question in one short sentence.",
    ),
    llms_config=OpenAILLMConfig(named_llm_config_id=saved.id),
)

ref_workflow: Workflow = await client.workflows.create_from_config(
    WorkflowConfigFullyHydrated(
        workflow_config=WorkflowConfig(
            name="Saved-config demo (10_llm_configs)",
            description="Single LLM node that resolves its provider settings from a saved config",
        ),
        node_configs=[assistant_node],
        edge_configs=[],
    )
)
REF_WF_ID = ref_workflow.id
print(f"Created workflow id={REF_WF_ID}  (its node's LLM → saved config {saved.id})")

# Run one turn. The server resolves `named_llm_config_id` to the stored
# provider/model/secret at run time — no keys travel with the node.
chat = await client.workflows.handle(REF_WF_ID)
run_input = WorkflowRunInput(
    command=WorkflowCommand.START,
    thread_to_node_inputs={
        "0": NodesRunInputs(
            node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content="What is a co-payment?")])]
        )
    },
)
print("User: What is a co-payment?")
async for event in chat.arun(run_input):
    if isinstance(event, AssistantResponseEvent) and event.content:
        print(f"  Assistant: {event.content}")

## Cleanup

In [ ]:
# Delete the demo workflow, then the saved config it referenced.
await client.workflows.delete(REF_WF_ID)
await client.llm_configs.delete(saved.id)
print("cleaned up")

## See also

- Guide: [`../docs/guides/llm_configs.md`](../docs/guides/llm_configs.md)
- [`11_reusable_assets.ipynb`](11_reusable_assets.ipynb) — node libraries, templates, categories
- [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb) — where `llms_config` lives on a node